# Synthetic SaaS Data Generation for NovaCRM

**Project 04 -- Executive KPI Report** | Notebook 1 of 5

## Purpose

This notebook documents the methodology behind generating 24 months of realistic synthetic SaaS data for NovaCRM, a mid-market B2B SaaS company in the \$5M--\$12M ARR range. Rather than using real company data (which carries privacy and NDA constraints), we engineer synthetic data that:

1. **Follows real SaaS distributions** -- growth rates, churn curves, and NPS distributions match industry benchmarks from OpenView Partners and SaaStr.
2. **Includes injected events** -- a Q3 2024 pricing-change churn spike and a Q2 2025 product launch bump, creating the kind of anomalies executives actually ask about.
3. **Is fully reproducible** -- seeded with `np.random.default_rng(42)`.

### Key Questions Answered

- What distributions and parameters drive the synthetic data?
- How do the generated metrics compare to SaaS industry benchmarks?
- Where are the injected business events visible in the data?
- How do the three customer segments (Starter, Professional, Enterprise) differ?

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# NovaCRM dashboard color palette
COLORS = {
    "cyan": "#06B6D4",
    "violet": "#8B5CF6",
    "emerald": "#10B981",
    "amber": "#F59E0B",
    "rose": "#F43F5E",
}
SEGMENT_COLORS = {
    "Starter": COLORS["cyan"],
    "Professional": COLORS["violet"],
    "Enterprise": COLORS["emerald"],
}

# Load generated data
monthly = pd.read_parquet("../data/processed/monthly_metrics.parquet")
segments = pd.read_parquet("../data/processed/segment_metrics.parquet")

print(f"monthly_metrics: {monthly.shape[0]} rows x {monthly.shape[1]} columns")
print(f"segment_metrics: {segments.shape[0]} rows x {segments.shape[1]} columns")
print(f"\nDate range: {monthly['month'].min().strftime('%b %Y')} to {monthly['month'].max().strftime('%b %Y')}")
print(f"Segments: {segments['segment'].unique().tolist()}")

## 1. Data Generation Methodology

The generation script (`data-pipeline/01_generate_saas_data.py`) uses a **state-machine approach**: each month's metrics evolve from the previous month's state, mimicking how real SaaS businesses grow incrementally.

### Starting State

| Metric | Initial Value | Rationale |
|--------|--------------|-----------|
| MRR | \$420,000 | Mid-market SaaS (~\$5M ARR) |
| Total Customers | 310 | Average ACV ~\$16K implies ~310 logos |
| NPS | 45 | Healthy but not exceptional |
| DAU | 4,800 | ~15 users/account on average |
| MAU | 14,500 | DAU/MAU ~0.33 (typical B2B) |

### Growth Model

Growth decelerates naturally from ~4% MoM early in 2024 to ~1.5% MoM by end of 2025, following the well-documented SaaS S-curve.

In [ ]:
# Reconstruct the growth rate curve used in generation
n_months = 24
base_growth = [0.040 - (i / n_months) * 0.025 for i in range(n_months)]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=monthly["month"],
    y=base_growth,
    mode="lines+markers",
    line=dict(color=COLORS["cyan"], width=3),
    marker=dict(size=6),
    name="Base Growth Rate",
))
fig.add_hline(y=0.03, line_dash="dash", line_color=COLORS["amber"],
              annotation_text="SaaStr median MoM growth (Series A)", annotation_position="top right")
fig.update_layout(
    title="Base MoM Growth Rate: Decelerating S-Curve",
    xaxis_title="Month",
    yaxis_title="Growth Rate",
    yaxis_tickformat=".1%",
    template="plotly_white",
    height=400,
    font=dict(family="Inter, sans-serif"),
)
fig.show()

## 2. Seasonality Pattern

New business acquisition follows a seasonal pattern common in B2B SaaS:

- **Q1 (Jan--Mar)**: Strong -- enterprise budget flush from new fiscal years (multiplier ~1.10x)
- **Q2 (Apr--Jun)**: Normal (multiplier ~1.00x)
- **Q3 (Jul--Sep)**: Summer dip (multiplier ~0.80x)
- **Q4 (Oct--Dec)**: December holiday stall drives Q4 down (December multiplier 0.65x)

This mirrors [OpenView's benchmarks](https://openviewpartners.com/blog/saas-benchmarks/) showing 15--25% seasonal swings in new logo acquisition.

In [ ]:
# Compute actual seasonal factors from the data
monthly["mom_mrr_growth"] = monthly["mrr"].pct_change()
monthly["quarter"] = monthly["month"].dt.quarter
monthly["month_name"] = monthly["month"].dt.strftime("%b")

# Seasonal pattern from new MRR
seasonal_avg = monthly.groupby("quarter")["new_mrr"].mean().reset_index()
seasonal_avg.columns = ["Quarter", "Avg New MRR"]

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "New MRR by Quarter (Avg)", "Seasonal Effect on New Business"
))

# Bar chart of quarterly averages
fig.add_trace(go.Bar(
    x=[f"Q{q}" for q in seasonal_avg["Quarter"]],
    y=seasonal_avg["Avg New MRR"],
    marker_color=[COLORS["emerald"], COLORS["cyan"], COLORS["amber"], COLORS["rose"]],
    text=[f"${v:,.0f}" for v in seasonal_avg["Avg New MRR"]],
    textposition="outside",
), row=1, col=1)

# Time series of new MRR with seasonal highlights
fig.add_trace(go.Scatter(
    x=monthly["month"], y=monthly["new_mrr"],
    mode="lines+markers",
    line=dict(color=COLORS["violet"], width=2),
    name="New MRR",
), row=1, col=2)

# Highlight summer dips
for year in [2024, 2025]:
    fig.add_vrect(
        x0=f"{year}-06-01", x1=f"{year}-09-01",
        fillcolor=COLORS["amber"], opacity=0.1,
        annotation_text="Summer dip" if year == 2024 else None,
        row=1, col=2,
    )
    fig.add_vrect(
        x0=f"{year}-12-01", x1=f"{year}-12-31",
        fillcolor=COLORS["rose"], opacity=0.1,
        annotation_text="Holiday" if year == 2024 else None,
        row=1, col=2,
    )

fig.update_layout(
    height=400, template="plotly_white", showlegend=False,
    title="Seasonality in New Business Acquisition",
    font=dict(family="Inter, sans-serif"),
)
fig.update_yaxes(tickprefix="$", tickformat=",", row=1, col=1)
fig.update_yaxes(tickprefix="$", tickformat=",", row=1, col=2)
fig.show()

## 3. Injected Business Events

Two events are deliberately injected to create the kind of anomalies that make dashboards valuable:

### Event 1: Q3 2024 Pricing Change (Churn Spike)
- **Months affected**: Jul--Sep 2024
- **Severity curve**: Jul (full impact) -> Aug (60%) -> Sep (25%)
- **Effects**: +2% added to churn rate, +0.5% to contraction rate, -10 NPS points, +25% support tickets

### Event 2: Q2 2025 Product Launch (Growth Bump)
- **Months affected**: Apr--Jun 2025
- **Severity curve**: Apr (60%) -> May (peak) -> Jun (70%)
- **Effects**: +40% new MRR boost, +70% expansion, -\$1,500 CAC improvement, +8 NPS points

> **So What?** These events are the "stories" in the data. An executive dashboard that surfaces them automatically provides more value than one that just shows flat trend lines.

In [ ]:
# Visualize the two injected events
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Logo Churn Rate (Pricing Event = shaded)",
        "NPS Score (Both Events)",
        "New MRR (Product Launch = shaded)",
        "Support Tickets (Pricing Event = shaded)",
    ),
)

# Churn rate
fig.add_trace(go.Scatter(
    x=monthly["month"], y=monthly["logo_churn_rate"],
    mode="lines+markers", line=dict(color=COLORS["rose"], width=2),
    name="Logo Churn",
), row=1, col=1)
fig.add_hline(y=0.03, line_dash="dash", line_color="gray",
              annotation_text="Target: 3%", row=1, col=1)

# NPS
fig.add_trace(go.Scatter(
    x=monthly["month"], y=monthly["nps"],
    mode="lines+markers", line=dict(color=COLORS["violet"], width=2),
    name="NPS",
), row=1, col=2)

# New MRR
fig.add_trace(go.Scatter(
    x=monthly["month"], y=monthly["new_mrr"],
    mode="lines+markers", line=dict(color=COLORS["emerald"], width=2),
    name="New MRR",
), row=2, col=1)

# Support tickets
fig.add_trace(go.Scatter(
    x=monthly["month"], y=monthly["support_tickets"],
    mode="lines+markers", line=dict(color=COLORS["amber"], width=2),
    name="Support Tickets",
), row=2, col=2)

# Shade pricing event (Q3 2024)
for row, col in [(1,1), (1,2), (2,2)]:
    fig.add_vrect(x0="2024-07-01", x1="2024-09-30",
                  fillcolor=COLORS["rose"], opacity=0.12,
                  annotation_text="Pricing change" if row == 1 and col == 1 else None,
                  row=row, col=col)

# Shade product launch (Q2 2025)
for row, col in [(1,2), (2,1)]:
    fig.add_vrect(x0="2025-04-01", x1="2025-06-30",
                  fillcolor=COLORS["emerald"], opacity=0.12,
                  annotation_text="Product launch" if row == 2 and col == 1 else None,
                  row=row, col=col)

fig.update_layout(
    height=600, template="plotly_white", showlegend=False,
    title="Injected Business Events and Their Effects",
    font=dict(family="Inter, sans-serif"),
)
fig.update_yaxes(tickformat=".1%", row=1, col=1)
fig.update_yaxes(tickprefix="$", tickformat=",", row=2, col=1)
fig.show()

## 4. Validation Against SaaS Industry Benchmarks

We compare the generated data against published benchmarks to ensure realism:

| Metric | NovaCRM (Generated) | Benchmark Range | Source |
|--------|-------------------|-----------------|--------|
| MRR Growth (MoM) | 1.5%--4.0% | 1%--5% | SaaStr Series A medians |
| Logo Churn (monthly) | 2.0%--4.5% | 2%--5% | OpenView 2024 |
| NRR (annualized) | 96%--115% | 100%--130% | Bessemer Cloud Index |
| NPS | 32--55 | 30--60 | SaaS NPS benchmarks |
| LTV:CAC | 3x--7x | 3x--5x (good) | a16z SaaS metrics |
| DAU/MAU | 0.28--0.38 | 0.25--0.40 | Mixpanel benchmarks |
| Gross Margin | 74%--86% | 70%--85% | KeyBanc SaaS Survey |

In [ ]:
# Distribution histograms for key metrics
metrics_to_plot = [
    ("mrr", "MRR ($)", True),
    ("logo_churn_rate", "Logo Churn Rate", False),
    ("nrr", "Net Revenue Retention", False),
    ("nps", "NPS Score", False),
    ("ltv_cac_ratio", "LTV:CAC Ratio", False),
    ("gross_margin", "Gross Margin", False),
]

fig = make_subplots(rows=2, cols=3, subplot_titles=[m[1] for m in metrics_to_plot])

colors_list = [COLORS["cyan"], COLORS["rose"], COLORS["emerald"],
               COLORS["violet"], COLORS["amber"], COLORS["cyan"]]

for idx, (col, label, is_currency) in enumerate(metrics_to_plot):
    row = idx // 3 + 1
    col_pos = idx % 3 + 1
    fig.add_trace(go.Histogram(
        x=monthly[col],
        nbinsx=10,
        marker_color=colors_list[idx],
        opacity=0.8,
        name=label,
    ), row=row, col=col_pos)

fig.update_layout(
    height=500, template="plotly_white", showlegend=False,
    title="Distribution of Key SaaS Metrics (24 months)",
    font=dict(family="Inter, sans-serif"),
)
fig.show()

# Print summary stats
print("Summary Statistics:")
print("=" * 70)
for col, label, _ in metrics_to_plot:
    vals = monthly[col]
    print(f"  {label:25s}  min={vals.min():>12,.2f}  median={vals.median():>12,.2f}  max={vals.max():>12,.2f}")

## 5. Segment Architecture

NovaCRM serves three customer segments with distinct economic profiles:

| Segment | % of Customers | % of MRR | Churn Baseline | NPS Baseline | Avg CAC |
|---------|---------------|----------|----------------|-------------|---------|
| Starter | ~55% | ~15% | 5.5% | 32 | ~\$3,000 |
| Professional | ~30% | ~40% | 3.0% | 45 | ~\$8,000 |
| Enterprise | ~15% | ~45% | 1.2% | 55 | ~\$18,000 |

> **So What?** Enterprise contributes 3x the revenue of Starter with 3.7x fewer logos -- classic SaaS unit economics where enterprise accounts dominate value despite being a small fraction of the logo count.

In [ ]:
# Segment comparison: latest month
latest_month = segments["month"].max()
latest_seg = segments[segments["month"] == latest_month].copy()

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("MRR by Segment", "Logo Churn Rate", "NPS Score"),
)

for i, metric in enumerate(["mrr", "logo_churn_rate", "nps"]):
    fig.add_trace(go.Bar(
        x=latest_seg["segment"],
        y=latest_seg[metric],
        marker_color=[SEGMENT_COLORS[s] for s in latest_seg["segment"]],
        text=[f"${v:,.0f}" if metric == "mrr" else f"{v:.1%}" if "rate" in metric else f"{v:.0f}"
              for v in latest_seg[metric]],
        textposition="outside",
        name=metric,
    ), row=1, col=i+1)

fig.update_layout(
    height=400, template="plotly_white", showlegend=False,
    title=f"Segment Comparison -- {latest_month.strftime('%B %Y')}",
    font=dict(family="Inter, sans-serif"),
)
fig.update_yaxes(tickprefix="$", tickformat=",", row=1, col=1)
fig.update_yaxes(tickformat=".1%", row=1, col=2)
fig.show()

In [ ]:
# MRR evolution by segment over time
fig = go.Figure()

for seg in ["Starter", "Professional", "Enterprise"]:
    seg_data = segments[segments["segment"] == seg]
    fig.add_trace(go.Scatter(
        x=seg_data["month"],
        y=seg_data["mrr"],
        mode="lines+markers",
        name=seg,
        line=dict(color=SEGMENT_COLORS[seg], width=2),
        marker=dict(size=5),
    ))

fig.update_layout(
    title="MRR Evolution by Customer Segment (Jan 2024 -- Dec 2025)",
    xaxis_title="Month",
    yaxis_title="MRR ($)",
    yaxis_tickprefix="$",
    yaxis_tickformat=",",
    template="plotly_white",
    height=450,
    legend=dict(x=0.02, y=0.98),
    font=dict(family="Inter, sans-serif"),
)
fig.show()

# Enterprise share of total MRR
latest = segments[segments["month"] == segments["month"].max()]
total_mrr = latest["mrr"].sum()
ent_share = latest[latest["segment"] == "Enterprise"]["mrr"].values[0] / total_mrr
print(f"\nEnterprise share of total MRR: {ent_share:.1%}")
print(f"Enterprise contributes ${latest[latest['segment'] == 'Enterprise']['mrr'].values[0]:,.0f} "
      f"of ${total_mrr:,.0f} total MRR")

## Summary

The synthetic data generation pipeline produces realistic SaaS metrics that:

1. **Match industry benchmarks** across all key dimensions (MRR growth, churn, NPS, unit economics)
2. **Include natural seasonality** that mirrors real B2B buying cycles
3. **Contain two deliberate business events** that test the anomaly detection system
4. **Differentiate across three segments** with economically coherent profiles

The data flows into notebook 02 (EDA) for deeper exploration and notebook 03 (anomaly detection) for automated alerting.

---
*Data generated by `data-pipeline/01_generate_saas_data.py` with seed `np.random.default_rng(42)`.*